In [ ]:
!apt-get install -y portaudio19-dev
!pip install SpeechRecognition pyttsx3 wikipedia requests pyaudio

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
portaudio19-dev is already the newest version (19.6.0-1.2build3).
0 upgraded, 0 newly installed, 0 to remove and 0 not upgraded.


In [ ]:
!pip install gTTS

from gtts import gTTS
from IPython.display import Audio, display

def falar(texto, lang='pt'):
    # Gera o arquivo de áudio usando a API do Google
    tts = gTTS(text=texto, lang=lang)
    filename = "voz.mp3"
    tts.save(filename)
    # Exibe o player de áudio diretamente no Colab
    display(Audio(filename, autoplay=True))

falar("Olá, eu sou sua assistente virtual.")

In [ ]:
import speech_recognition as sr
from google.colab import output
import base64
import io

# Instala o pydub silenciosamente caso não esteja instalado
!pip install pydub -q
from pydub import AudioSegment

# Código em JavaScript otimizado para capturar áudio pelo navegador do usuário
RECORD_JS = """
const sleep = time => new Promise(resolve => setTimeout(resolve, time));
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader();
  reader.onloadend = () => resolve(reader.result);
  reader.readAsDataURL(blob);
});
var record = time => new Promise(async resolve => {
  navigator.mediaDevices.getUserMedia({ audio: true })
    .then(stream => {
      const mediaRecorder = new MediaRecorder(stream);
      mediaRecorder.start();
      const audioChunks = [];
      mediaRecorder.addEventListener("dataavailable", event => {
        audioChunks.push(event.data);
      });
      mediaRecorder.addEventListener("stop", async () => {
        const audioBlob = new Blob(audioChunks, { type: 'audio/webm' });
        const audioUrl = await b2text(audioBlob);
        resolve(audioUrl);
      });
      setTimeout(() => mediaRecorder.stop(), time);
    }).catch(err => {
      console.error("Erro ao acessar o microfone:", err);
      resolve(null);
    });
});
"""

def ouvir(segundos=5):
    print(f"\n[Microfone] Gravando por {segundos} segundos... FALE AGORA!")
    try:
        # Executa o script JS para capturar o áudio por 'segundos'
        display(output.eval_js(RECORD_JS))
        audio_base64 = output.eval_js(f"record({segundos * 1000})")

        if not audio_base64:
            print("[Erro] Permissão de microfone negada ou não detectada no navegador.")
            return ""

        # Decodifica o áudio em bytes (formato webm)
        audio_bytes = base64.b64decode(audio_base64.split(',')[1])

        # Converte o WebM para WAV usando o pydub
        audio_segment = AudioSegment.from_file(io.BytesIO(audio_bytes), format="webm")
        filename = "temp_audio.wav"
        audio_segment.export(filename, format="wav")

        # Utiliza o SpeechRecognition no arquivo WAV decodificado
        recognizer = sr.Recognizer()
        with sr.AudioFile(filename) as source:
            # Ajusta para ruídos ambientes
            recognizer.adjust_for_ambient_noise(source, duration=0.2)
            audio_data = recognizer.record(source)

        try:
            comando = recognizer.recognize_google(audio_data, language="pt-BR")
            print("-> Você disse:", comando)
            return comando.lower()
        except sr.UnknownValueError:
            print("[Aviso] Google Speech Recognition não conseguiu entender o áudio.")
            return ""
        except sr.RequestError as e:
            print(f"[Erro] Não foi possível solicitar resultados do Google Speech Recognition; {e}")
            return ""

    except Exception as e:
        print(f"[Erro inesperado na gravação]: {e}")
        return ""

In [ ]:
from IPython.display import Javascript, display

def abrir_youtube():
    display(Javascript('window.open("https://www.youtube.com", "_blank");'))

In [ ]:
from IPython.display import Javascript, display

def abrir_farmacia():
    display(Javascript('window.open("https://www.google.com/maps/search/farmacia+proxima", "_blank");'))

In [ ]:
import wikipedia

def pesquisar_wikipedia(tema):
    # Configura o idioma do Wikipédia para português
    wikipedia.set_lang("pt")
    try:
        # Busca o resumo da página referente ao tema com limite de 2 frases
        resumo = wikipedia.summary(tema, sentences=2)
        return resumo
    except wikipedia.exceptions.DisambiguationError as e:
        return f"O termo '{tema}' é muito genérico. Por favor, seja mais específico."
    except wikipedia.exceptions.PageError:
        return f"Não consegui encontrar nenhuma página sobre '{tema}' no Wikipédia."
    except Exception as e:
        return "Desculpe, ocorreu um erro ao realizar a busca."

In [ ]:
def executar_comando(comando):
    if "youtube" in comando:
        falar("Abrindo o YouTube")
        abrir_youtube()

    elif "farmácia" in comando:
        falar("Procurando farmácias próximas")
        abrir_farmacia()

    elif "pesquise" in comando:
        tema = comando.replace("pesquise", "")
        resultado = pesquisar_wikipedia(tema)
        print(resultado)
        falar(resultado)

    elif "sair" in comando:
        falar("Encerrando assistente")
        return False

    else:
        falar("Comando não reconhecido")

    return True

In [ ]:
def main():
    falar("Assistente virtual iniciada")
    ativo = True
    while ativo:
        comando = ouvir()
        if comando != "":
            ativo = executar_comando(comando)

if __name__ == "__main__":
    main()


[Microfone] Gravando por 5 segundos... FALE AGORA!


None

[Aviso] Google Speech Recognition não conseguiu entender o áudio.

[Microfone] Gravando por 5 segundos... FALE AGORA!


None

[Aviso] Google Speech Recognition não conseguiu entender o áudio.

[Microfone] Gravando por 5 segundos... FALE AGORA!


None

... (9 outputs left out)

None

[Aviso] Google Speech Recognition não conseguiu entender o áudio.

[Microfone] Gravando por 5 segundos... FALE AGORA!


None

-> Você disse: sair
